<a href="https://colab.research.google.com/github/ayushmann25bce10347-jpg/OmniChem-Universal-Chemical-Property-Extraction-Analysis-System/blob/main/OmniChem_Universal_Chemical_Property_Extraction_%26_Analysis_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors
import pubchempy as pcp
import os

class UniversalMoleculeAnalyzer:
    def __init__(self, molecule_name):
        self.name = molecule_name
        print(f"--- Fetching Data for: {self.name} ---")

        try:
            # Search for the compound
            compounds = pcp.get_compounds(molecule_name, 'name')
            if not compounds:
                raise ValueError(f"Molecule '{molecule_name}' not found.")

            self.compound = compounds[0]
            self.smiles = self.compound.isomeric_smiles
            self.mol = Chem.AddHs(Chem.MolFromSmiles(self.smiles))

        except Exception as e:
            print(f"Search Error: {e}")
            self.mol = None

    def get_full_report(self):
        if self.mol is None: return "No data available."

        # 1. Molecular Properties
        mol_data = {
            "Chemical Name": self.name.capitalize(),
            "Formula": self.compound.molecular_formula,
            "SMILES": self.smiles,
            "Exact Mass": round(Descriptors.ExactMolWt(self.mol), 4),
            "LogP": round(Descriptors.MolLogP(self.mol), 2),
            "TPSA": round(Descriptors.TPSA(self.mol), 2),
            "HBond Donors": Descriptors.NumHDonors(self.mol),
            "HBond Acceptors": Descriptors.NumHAcceptors(self.mol)
        }

        # 2. Atomic Breakdown
        atoms = []
        for atom in self.mol.GetAtoms():
            atoms.append({
                "Symbol": atom.GetSymbol(),
                "AtomicNum": atom.GetAtomicNum(),
                "Hybridization": atom.GetHybridization(),
                "FormalCharge": atom.GetFormalCharge()
            })

        return pd.Series(mol_data), pd.DataFrame(atoms)

# --- EXECUTION ---
if __name__ == "__main__":
    user_input = input("Enter any molecule name: ")
    analyzer = UniversalMoleculeAnalyzer(user_input)

    summary, atom_table = analyzer.get_full_report()

    print("\n[SUMMARY]")
    print(summary)
    print("\n[ATOMIC ARCHITECTURE]")
    print(atom_table.head(10)) # Shows first 10 atoms


Enter any molecule name: Glucose
--- Fetching Data for: Glucose ---

[SUMMARY]
Chemical Name                                         Glucose
Formula                                               C6H12O6
SMILES             C([C@@H]1[C@H]([C@@H]([C@H](C(O1)O)O)O)O)O
Exact Mass                                           180.0634
LogP                                                    -3.22
TPSA                                                   110.38
HBond Donors                                                5
HBond Acceptors                                             6
dtype: object

[ATOMIC ARCHITECTURE]
  Symbol  AtomicNum  Hybridization  FormalCharge
0      C          6              4             0
1      C          6              4             0
2      C          6              4             0
3      C          6              4             0
4      C          6              4             0
5      C          6              4             0
6      O          8              4           

/tmp/ipykernel_10568/338878301.py:19: PubChemPyDeprecationWarning: isomeric_smiles is deprecated: Use smiles instead
  self.smiles = self.compound.isomeric_smiles
